---
# Topic Modelling con BERTopic
## Python + Embeddings + Clustering
---

En **5a** ajustamos modelos LDA en R sobre el corpus de críticas de cine de **Pang & Lee (2004)** (`data_corpus_moviereviews` en `quanteda`), probando $k=12$, $k=20$ y $k=25$ tópicos.

En este notebook repetimos ese análisis con **BERTopic**, una librería de Python que descubre tópicos de una forma fundamentalmente distinta a LDA: en vez de modelar el corpus como una mezcla probabilística de distribuciones de palabras (Dirichlet), BERTopic:

1. Convierte cada documento en un **embedding** (un vector que captura su significado).
2. Reduce la dimensionalidad de esos vectores con **UMAP**.
3. Agrupa documentos semánticamente similares con **HDBSCAN** (*clustering*).
4. Describe cada grupo (tópico) con sus palabras más distintivas usando **c-TF-IDF**.

Usamos el mismo corpus (o una versión equivalente) que en 5a: `nltk.corpus.movie_reviews` distribuye el mismo conjunto de 2000 críticas de Pang & Lee que `quanteda` empaqueta como `data_corpus_moviereviews`, lo que nos permite comparar ambos enfoques sobre datos equivalentes.

> 🖥️ **Sobre el hardware:** generar embeddings para 2000 documentos toma más tiempo que embeber palabras sueltas (como en 5c), pero sigue siendo mucho más liviano que 5b — del orden de uno a pocos minutos, incluso en CPU. Usaremos GPU automáticamente si está disponible.

## 🎯 Objetivos de aprendizaje

- Entender conceptualmente en qué se diferencia BERTopic de LDA: *embeddings* + *clustering* en vez de un modelo generativo probabilístico.
- Ajustar un modelo BERTopic sobre el corpus de críticas de cine y explorar los tópicos que descubre **automáticamente** (sin especificar $k$ de antemano).
- Reducir el número de tópicos a una cantidad comparable con los modelos de 5a/5b/5c.
- Medir la coherencia semántica de los tópicos de BERTopic con el **mismo método de embeddings de 5c**, para comparar directamente la calidad de ambos enfoques.

## ⚖️ LDA vs. BERTopic: una diferencia clave

Algo que ya notamos en 5a, 5b y 5c, pero vale la pena hacer explícito: con LDA, **tú decides cuántos tópicos ($k$) quieres antes de ajustar el modelo** — de hecho, gran parte de 5a estuvo dedicada a intentar *adivinar* un buen valor de $k$ usando perplexity y el método del codo.

BERTopic invierte esta lógica: **descubre el número de tópicos automáticamente**, a partir de cuántos grupos naturales encuentra HDBSCAN en el espacio de embeddings. Esto tiene ventajas (no hay que adivinar $k$) y desventajas (menos control directo, y el número de tópicos puede variar entre ejecuciones o corpus).

> 📌 Más adelante en este notebook forzaremos a BERTopic a reducirse a un número específico de tópicos, para poder comparar su coherencia directamente con los modelos de $k$ fijo de 5a-5c.

## 📦 Instalación de las librerías necesarias

In [ ]:
!pip install -q bertopic nltk sentence-transformers pandas numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import nltk
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

## 📂 Cargar el corpus

Descargamos el corpus de críticas de cine de Pang & Lee vía NLTK — el mismo conjunto de 2000 documentos (1000 positivos, 1000 negativos) que usamos en R como `data_corpus_moviereviews`.

In [ ]:
nltk.download("movie_reviews")

from nltk.corpus import movie_reviews

documentos = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()]
print(f"Número de documentos: {len(documentos)}")
print()
print(documentos[0][:300])

## 🧠 Ajustar BERTopic

BERTopic usa por defecto `all-MiniLM-L6-v2`, un modelo de embeddings liviano y rápido, recomendado por la propia librería para esta etapa de *clustering* de documentos (distinto del modelo `all-mpnet-base-v2` que usamos en 5c para *medir* coherencia — aquí usamos el modelo estándar de BERTopic para descubrir los tópicos, y en la sección de evaluación reutilizaremos el de 5c para mantener la comparación de coherencia consistente con las unidades anteriores).

> ⚠️ **Importante — eliminar *stop words*:** a diferencia de LDA en 5a, donde aplicamos `tokens_remove(stopwords("en"))` explícitamente antes de ajustar el modelo, BERTopic **no elimina *stop words* por defecto** en la representación de sus tópicos (el paso de *c-TF-IDF* que decide qué palabras describen cada tópico). Sin este paso, los tópicos terminan dominados por palabras genéricas como "the", "and", "is" — sin valor temático, y muy distintas entre sí en frecuencia con respecto a las palabras de contenido real. Para que la comparación con LDA sea justa (misma limpieza de texto de ambos lados), le pasamos un `CountVectorizer` con `stop_words="english"` a través del parámetro `vectorizer_model`. Esto solo afecta a **qué palabras se muestran como representativas de cada tópico** — los embeddings usados para el *clustering* siguen calculándose sobre el texto completo de cada documento, sin este filtro.

Detectamos el hardware disponible igual que en notebooks anteriores.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Usando dispositivo: {device}")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
vectorizer_model = CountVectorizer(stop_words="english")
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="english",
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(documentos)

## 🔍 Explorar los tópicos descubiertos

`get_topic_info()` resume cada tópico encontrado: su identificador, cuántos documentos contiene, y sus palabras más representativas. El tópico `-1` es especial: agrupa los documentos que BERTopic considera **ruido** (no encajan claramente en ningún grupo) — no es un tópico temático real y normalmente se excluye de los análisis.

In [ ]:
info_topicos = topic_model.get_topic_info()
print(f"BERTopic encontró {len(info_topicos) - 1} tópicos (sin contar el tópico -1 de ruido)")
info_topicos.head(15)

### ⚠️ ¿Qué es el tópico `-1`?

Fíjate en la primera fila de la tabla anterior: el tópico `-1` no es un tema temático real — es el *bucket* de **ruido** de HDBSCAN, el algoritmo de *clustering* que usa BERTopic internamente.

A diferencia de un algoritmo como k-means (que fuerza a **cada** documento a pertenecer al clúster más cercano, sin importar qué tan mal encaje), HDBSCAN puede decidir que un documento **no pertenece claramente a ningún grupo denso**, y lo etiqueta como `-1` en vez de forzarlo a un tópico que no le corresponde bien.

En una ejecución real sobre este corpus, BERTopic descubrió **37 tópicos naturalmente** (antes de reducirlos a 20 para comparar con LDA) — y el tópico `-1` concentró **910 de las 2000 críticas (45.5% del corpus)**. Es un número alto: casi la mitad de las reseñas no encajaron claramente en ningún tema específico.

¿Por qué podría pasar esto con reseñas de películas? Las palabras representativas del tópico `-1` (`film, movie, like, just, time, good, character...`) son precisamente el vocabulario genérico que **cualquier** crítica de cine usa, sin importar de qué película se trate — lenguaje sobre el *acto de reseñar*, no sobre el contenido específico de una película. Muchas reseñas están escritas en un estilo lo bastante genérico como para no acercarse lo suficiente, en el espacio de embeddings, a ningún grupo temático específico.

> 📌 **Esto es una limitación real que vale la pena reconocer, no ocultar.** A diferencia de LDA — que asigna *alguna* mezcla de tópicos a cada documento, sin excepción — BERTopic con su configuración por defecto puede terminar sin clasificar casi la mitad del corpus. Ajustar el hiperparámetro `min_topic_size` de HDBSCAN (más pequeño → menos ruido, pero tópicos menos "limpios"; más grande → lo opuesto) es una forma de explorar este *trade-off*, aunque queda fuera del alcance de este notebook.

In [ ]:
n_ruido = info_topicos.loc[info_topicos["Topic"] == -1, "Count"].values[0]
n_total = info_topicos["Count"].sum()

print(f"Documentos sin tópico asignado (ruido): {n_ruido} de {n_total} ({n_ruido / n_total:.1%})")

In [ ]:
# palabras más representativas de un par de tópicos, a modo de ejemplo
topic_model.get_topic(0)

## 📊 Visualizar los tópicos

BERTopic incluye visualizaciones interactivas listas para usar — útiles para explorar los resultados sin tener que armar los gráficos manualmente, como sí hicimos en 5a con `ggplot2`.

In [ ]:
topic_model.visualize_barchart(top_n_topics=12)

## 🔧 Reducir a un número comparable de tópicos

Para comparar la coherencia de BERTopic con los modelos de $k=20$ de 5a-5c en igualdad de condiciones, reducimos BERTopic a 20 tópicos con `reduce_topics()`.

### 📐 ¿Qué es TF-IDF?

**TF-IDF** (*Term Frequency – Inverse Document Frequency*) es una forma clásica de medir qué tan **distintiva** es una palabra para un documento específico, dentro de una colección de documentos.

La idea combina dos partes:

$$
\text{tf}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}
$$

**Frecuencia del término (tf):** qué tan seguido aparece la palabra $t$ en el documento $d$, respecto al total de palabras de ese documento.

$$
\text{idf}(t, D) = \log\left(\frac{N}{|\{d \in D : t \in d\}|}\right)
$$

**Frecuencia inversa de documento (idf):** el logaritmo del número total de documentos $N$ dividido por el número de documentos que contienen la palabra $t$. Palabras que aparecen en **casi todos** los documentos (como "the", "movie", "film" en nuestro corpus) tienen un `idf` cercano a cero — no distinguen nada. Palabras raras, presentes solo en unos pocos documentos, tienen un `idf` alto.

$$
\text{tfidf}(t, d, D) = \text{tf}(t, d) \times \text{idf}(t, D)
$$

El puntaje final es el producto de ambas: una palabra tiene un TF-IDF alto si aparece **frecuentemente en un documento** pero **raramente en el resto de la colección** — exactamente el tipo de palabra que ya vimos que le faltaba al tópico `-1` (dominado por vocabulario genérico de crítica de cine, con TF-IDF bajo porque aparece en casi todas partes).

> 📚 **Referencia:** [Wikipedia — tf–idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf)

### 🧩 c-TF-IDF: la variante que usa BERTopic

BERTopic no calcula TF-IDF por documento individual, sino una variante llamada **c-TF-IDF** (*class-based TF-IDF*, propuesta en el paper original de BERTopic): trata a **todos los documentos de un mismo tópico como si fueran un solo documento combinado**, y calcula qué palabras distinguen a ese tópico frente a los demás tópicos (en vez de qué palabras distinguen a un documento frente a los demás documentos).

$$
W_{t,c} = \text{tf}(t, c) \times \log\left(1 + \frac{A}{f_t}\right)
$$

donde $\text{tf}(t, c)$ es la frecuencia del término $t$ dentro de la clase (tópico) $c$, $A$ es el número promedio de palabras por clase, y $f_t$ es la frecuencia del término $t$ sumada sobre **todas** las clases. La lógica es la misma que en TF-IDF clásico, solo que "documento" pasa a ser "tópico completo".

Esta es la representación que ya usamos indirectamente: son las palabras de c-TF-IDF las que aparecen en `get_topic()` y en la tabla de `get_topic_info()`.

> 📚 **Referencia:** Grootendorst, M. (2022). *BERTopic: Neural topic modeling with a class-based TF-IDF procedure*. [arXiv:2203.05794](https://arxiv.org/abs/2203.05794)

### 🔀 ¿Cómo decide `reduce_topics()` qué tópicos fusionar?

`reduce_topics()` no vuelve a ejecutar el *clustering* (HDBSCAN) con otros parámetros para llegar "naturalmente" a 20 tópicos. En cambio, fusiona los tópicos existentes de forma iterativa:

1. Calcula la **similitud coseno entre los vectores c-TF-IDF** de cada par de tópicos — es decir, qué tan parecido es el "perfil de palabras distintivas" de un tópico con el de otro.
2. Encuentra el par de tópicos **más similares** entre sí y los fusiona en uno solo (los documentos de ambos quedan bajo el tópico fusionado, y se recalcula su representación c-TF-IDF).
3. Repite este proceso — encontrar el par más similar, fusionar — hasta llegar al número de tópicos solicitado (`nr_topics=20`).

El criterio es entonces **redundancia**: los tópicos que se fusionan primero son los que ya se parecían más entre sí en su vocabulario distintivo, no los más pequeños, ni los que un humano consideraría menos interesantes. El tópico `-1` (ruido) queda **excluido** de este proceso — nunca se fusiona con nada, y conserva exactamente sus mismos documentos.

> 📌 Existe también la opción `nr_topics="auto"`, que usa este mismo mecanismo de fusión pero se detiene sola (según un umbral de similitud) en vez de forzar un número exacto — una alternativa más "orgánica" a la que usamos aquí, donde elegimos 20 deliberadamente para poder comparar con LDA.

In [ ]:
topic_model.reduce_topics(documentos, nr_topics=20)

info_reducida = topic_model.get_topic_info()
print(f"Tópicos después de reducir: {len(info_reducida) - 1}")
info_reducida.head(21)

## 📐 Medir la coherencia con el método de 5c

Reutilizamos exactamente la misma función de 5c: para cada tópico, tomamos sus 15 palabras más representativas, las convertimos en embeddings con `all-mpnet-base-v2`, y promediamos la similitud coseno entre todos los pares de palabras.

Usar el **mismo modelo y método de evaluación** que en 5c es lo que hace válida la comparación: cualquier diferencia en el puntaje de coherencia refleja una diferencia real entre cómo LDA y BERTopic agrupan las palabras, no un artefacto de usar métricas distintas.

In [ ]:
modelo_embeddings = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)


def coherencia_por_embeddings(palabras):
    embeddings = modelo_embeddings.encode(palabras)
    matriz_similitud = cosine_similarity(embeddings)
    n = len(palabras)
    mascara = ~np.eye(n, dtype=bool)
    similitudes = matriz_similitud[mascara]
    return similitudes.mean()

In [ ]:
resultados_bertopic = []
for topic_id in info_reducida["Topic"]:
    if topic_id == -1:
        continue  # excluimos el tópico de ruido
    palabras = [palabra for palabra, _ in topic_model.get_topic(topic_id)][:15]
    coherencia = coherencia_por_embeddings(palabras)
    resultados_bertopic.append({
        "topic": topic_id,
        "coherencia_embeddings": coherencia,
        "palabras": ", ".join(palabras),
    })

resultados_bertopic = pd.DataFrame(resultados_bertopic)
resultados_bertopic

## 🆚 Comparar: BERTopic vs. LDA ($k=20$)

En 5c, la coherencia promedio de LDA con $k=20$ fue de aproximadamente **0.258**. Comparemos ese número con el promedio de BERTopic.

In [ ]:
coherencia_promedio_bertopic = resultados_bertopic["coherencia_embeddings"].mean()
coherencia_promedio_lda_k20 = 0.258  # de 5c

print(f"Coherencia promedio — BERTopic (20 tópicos): {coherencia_promedio_bertopic:.4f}")
print(f"Coherencia promedio — LDA, k=20 (desde 5c):   {coherencia_promedio_lda_k20:.4f}")

plt.figure(figsize=(5, 4))
plt.bar(["LDA (k=20)", "BERTopic (20 tópicos)"], [coherencia_promedio_lda_k20, coherencia_promedio_bertopic], color=["#175895", "#854f0b"])
plt.ylabel("Similitud coseno promedio (embeddings)")
plt.title("Coherencia semántica: LDA vs. BERTopic")
plt.tight_layout()
plt.show()

> 📌 **Antes de sacar conclusiones:** ejecuta la celda anterior y observa el resultado real. El punto de este notebook no es asumir de antemano que BERTopic "gana" — es comparar ambos números con el mismo método de medición y discutir **por qué** podrían diferir (o no). Si BERTopic obtiene un puntaje más alto, es evidencia a favor de la idea (mencionada en el programa del curso) de que los embeddings mejoran la coherencia temática frente al enfoque bag-of-words de LDA. Si no, también es un resultado válido y interesante para discutir — quizás con este corpus y este número de tópicos, ambos métodos capturan una estructura temática similar.

### 🔍 Lo que muestran los resultados

En una ejecución real (después de corregir el filtrado de *stop words*), BERTopic obtuvo:

| Enfoque | Coherencia promedio (embeddings) |
|---|---|
| LDA, $k=20$ (desde 5c) | ≈ 0.258 |
| BERTopic (19 tópicos tras reducir) | ≈ 0.266 |

La diferencia en el promedio es real pero **modesta** (~3% más alto) — no una mejora dramática. Sin embargo, hay algo más interesante que el promedio: revisando las palabras de cada tópico, BERTopic produjo agrupaciones notablemente **más específicas** que LDA. Mientras los tópicos de LDA en 5a tendían a describir géneros amplios ("Sci-Fi Action", "Disney Animation", "Horror Teen Movies"), varios tópicos de BERTopic corresponden a **películas o franquicias individuales**: *Star Wars* (`wars, star, jedi, phantom, lucas`), *Tarzan* (`tarzan, apes, ape, planet, jane, gorilla`), *Titanic* (`titanic, ship, rose, cameron, winslet`), *Batman* (`batman, robin, supergirl, ivy, freeze`), *El Show de Truman* (`truman, carrey, burbank, weir, christof`).

Esto sugiere que los embeddings de documentos completos capturan una noción de similitud más **fina** que la coocurrencia de palabras que usa LDA — suficiente para separar películas específicas dentro de un mismo género, no solo géneros entre sí. Es una evidencia parcial a favor de la idea del programa del curso ("los embeddings mejoran la coherencia temática"), aunque el efecto se ve más claramente en la **especificidad temática** de los tópicos que en el puntaje numérico promedio.

> 📌 **Idea clave:** un promedio de coherencia similar no significa que ambos métodos produzcan el mismo tipo de agrupación. Vale la pena mirar más allá del número y leer las palabras — la métrica agregada y la inspección cualitativa cuentan historias distintas, y ambas son necesarias.

## ✅ Cierre

Con este notebook completamos un recorrido por **cuatro formas distintas** de abordar el modelamiento de tópicos sobre el mismo corpus:

- **5a — LDA (R / Quanteda)**: modelo generativo probabilístico clásico, interpretable y reproducible, pero requiere elegir $k$ de antemano.
- **5b — LLM (Python)**: un modelo de lenguaje lee y etiqueta los tópicos de LDA, aportando comprensión semántica cualitativa — con la limitación de que sus puntajes de coherencia (escala 1-5) resultaron poco discriminantes.
- **5c — Embeddings (Python)**: mide la coherencia de los tópicos de LDA matemáticamente, con similitud coseno — más granular que el LLM, pero con poca capacidad para distinguir entre distintos valores de $k$.
- **5d — BERTopic (Python)**: un enfoque de modelamiento de tópicos completamente distinto, basado en embeddings y *clustering* en vez de en un modelo probabilístico — determina el número de tópicos automáticamente, y evaluamos su coherencia con la misma métrica de 5c para comparar de forma justa contra LDA.

> 📌 **Idea clave del bloque completo (5a-5d):** no existe un único método "correcto" para modelar y evaluar tópicos. Cada enfoque — estadístico, basado en LLM, basado en embeddings, o un algoritmo de *clustering* alternativo — responde una pregunta ligeramente distinta y tiene sus propias fortalezas y limitaciones. Comparar sus resultados de forma crítica, en vez de confiar ciegamente en uno solo, es la habilidad central que buscamos desarrollar en este curso.